# The Engineer's Crash Course to DeepChem 🧪🤖

Welcome! This notebook is a working demonstration of **DeepChem**, utilizing a PyTorch **Graph Convolutional Network (GCN)** running on your CPU.

*(Note: When you are ready to test your DGL Apple Silicon rewrite, change `device='cpu'` to `device='mps'` in the Model block!)*
---
### The Goal: Predicting Toxicity
We are going to train an AI to look at the structure of a chemical molecule and predict whether it is toxic to humans. Specifically, we are using the **Tox21 Dataset** (Toxicology in the 21st Century), a public database of 8,000+ chemicals tested against 12 different biological targets (like the Androgen Receptor or p53 pathways).

In [1]:
import deepchem as dc
import torch

print("PyTorch Version:", torch.__version__)
print("DeepChem Version:", dc.__version__)

No normalization for SPS. Feature removed!


No normalization for AvgIpc. Feature removed!


No normalization for NumAmideBonds. Feature removed!


No normalization for NumAtomStereoCenters. Feature removed!


No normalization for NumBridgeheadAtoms. Feature removed!


...

cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/opt/homebrew/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)


Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/opt/homebrew/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)


Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'


Skipped loading some Jax models, missing a dependency. No module named 'jax'


PyTorch Version: 2.2.0
DeepChem Version: 2.8.0


### Step 1: Featurization (Turning Chemistry into Math) ⚛️➡️🔢

Machine learning models don't understand 3D molecules. They only understand matrices of numbers. The process of converting a chemical into a mathematical representation is called **Featurization**.

In cheminformatics, molecules are usually stored as **SMILES strings** (e.g., `C1=CC=CC=C1` for Benzene). DeepChem reads these strings and converts them.

**The Graph Convolution Approach:**
Instead of flattening the molecule into a 1D array, a Graph Convolutional Network (GCN) treats the molecule as a 2D Network Graph:
- **Atoms** become **Nodes** (with data attached, like atomic number, charge, hybridization).
- **Bonds** become **Edges** (with data attached, like single/double/triple bond).

Here, we use `MolGraphConvFeaturizer()` to parse the Tox21 dataset into PyTorch `GraphData` objects.

In [2]:
featurizer = dc.feat.MolGraphConvFeaturizer()
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer)
train_dataset, valid_dataset, test_dataset = datasets

print("Number of biological targets to predict:", len(tasks))
print("Example Biological Targets:", tasks[:3])

Number of biological targets to predict: 12
Example Biological Targets: ['NR-AR', 'NR-AR-LBD', 'NR-AhR']


### Step 2: The Graph Convolutional Network (GCN) 🧠

How does a GCN actually learn chemistry?
It uses "Message Passing". In each layer of the neural network, an atom "talks" to its immediate neighbors (the atoms it is bonded to) and updates its own mathematical state based on what they say. 
After a few layers (convolutions), every atom has a mathematical understanding of its local chemical environment. The model then sums up all the atoms to create a "Fingerprint" of the entire molecule, and uses that to predict toxicity.

*(Below, we initialize the model on the CPU. Change `cpu` to `mps` when you are ready to test your Metal C++ port!)*

# NOTE: To test your DGL Apple Silicon rewrite, change 'cpu' to 'mps' below!
device = torch.device('cpu')

model = dc.models.GCNModel(
    n_tasks=len(tasks), 
    mode='classification', 
    dropout=0.2, 
    device=device
)
print("PyTorch Model Initialized on:", model.device)

### Step 3: Train the AI 🏋️‍♂️

We call `model.fit()` to pass the chemical graphs through the network. The AI makes predictions, calculates the error against the actual lab results, and adjusts its weights to get smarter.

print("Training the AI on the chemical data... This may take a minute on CPU.")
model.fit(train_dataset, nb_epoch=10)

### Step 4: Grading the AI 🎓

We evaluate the model using the **ROC-AUC** metric (Area Under the Receiver Operating Characteristic Curve). 
- `0.5` means the AI is randomly guessing.
- `1.0` means the AI is a perfect oracle.
- Usually, anything above `0.75` on the Test Set is considered highly predictive for biology!

metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

train_score = model.evaluate(train_dataset, [metric], transformers)
test_score = model.evaluate(test_dataset, [metric], transformers)

print('Training Set Score:', train_score['roc_auc_score'])
print('Test Set Score:', test_score['roc_auc_score'])

## 5. Performance Benchmark: CPU vs. Apple MPS GPU 🚀

Let's benchmark GCNModel training on your CPU vs. Apple Silicon GPU (MPS) and compare the durations and samples trained per second (throughput).


import time

num_samples = len(train_dataset)
print(f"Number of training samples: {num_samples}\n")

# 1. Benchmark CPU
print("--- Benchmarking CPU ---")
device_cpu = torch.device('cpu')
model_cpu = dc.models.GCNModel(
    n_tasks=len(tasks), 
    mode='classification', 
    dropout=0.2, 
    device=device_cpu
)
model_cpu.fit(train_dataset, nb_epoch=1) # Warmup

start_cpu = time.time()
model_cpu.fit(train_dataset, nb_epoch=3)
end_cpu = time.time()
duration_cpu = end_cpu - start_cpu
samples_sec_cpu = (num_samples * 3) / duration_cpu
print(f"CPU Total (3 epochs): {duration_cpu:.2f}s | Avg Epoch: {duration_cpu/3:.2f}s | Throughput: {samples_sec_cpu:.2f} samples/sec\n")

# 2. Benchmark MPS GPU
print("--- Benchmarking MPS GPU ---")
device_mps = torch.device('mps')
model_mps = dc.models.GCNModel(
    n_tasks=len(tasks), 
    mode='classification', 
    dropout=0.2, 
    device=device_mps
)
model_mps.fit(train_dataset, nb_epoch=1) # Warmup

start_mps = time.time()
model_mps.fit(train_dataset, nb_epoch=3)
end_mps = time.time()
duration_mps = end_mps - start_mps
samples_sec_mps = (num_samples * 3) / duration_mps
print(f"MPS Total (3 epochs): {duration_mps:.2f}s | Avg Epoch: {duration_mps/3:.2f}s | Throughput: {samples_sec_mps:.2f} samples/sec\n")

# 3. Print Speedup
print("--- Comparison ---")
print(f"MPS GPU Speedup over CPU: {samples_sec_mps / samples_sec_cpu:.2f}x")


## 6. Dense GCN Benchmark (GPU vs. CPU, 2.9 GB RAM Peak)

Let's stress-test training with a Dense GCN model using **1,800 hidden channels** and **full-batch training** (processing all 6,245 molecules in a single forward/backward pass). This maximizes the compute load to stress-test Apple Silicon GPU acceleration while staying safely under your **3 GB memory limit**.


In [3]:
# Dense GCN Benchmark (Full-Batch, Memory Limit: 2.88 GB, In-Place Optimizations)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

# Strictly limit CPU thread count to keep your system responsive
torch.set_num_threads(4)

print("Preparing dataset for full-batch Dense GCN...")
max_nodes = 60
X_list = []
y_list = []
w_list = []

for X, y, w, id in train_dataset.itersamples():
    X_list.append(X)
    y_list.append(y)
    w_list.append(w)

batch_size = len(X_list)
dense_X = torch.zeros(batch_size, max_nodes, 30)
dense_Adj = torch.zeros(batch_size, max_nodes, max_nodes)
dense_y = torch.from_numpy(np.array(y_list)).float()
dense_w = torch.from_numpy(np.array(w_list)).float()

for i, g in enumerate(X_list):
    num_nodes = min(g.num_nodes, max_nodes)
    dense_X[i, :num_nodes, :] = torch.from_numpy(g.node_features[:num_nodes, :]).float()
    edge_index = g.edge_index
    for u, v in zip(edge_index[0], edge_index[1]):
        if u < max_nodes and v < max_nodes:
            dense_Adj[i, u, v] = 1.0
            dense_Adj[i, v, u] = 1.0
    for u in range(num_nodes):
        dense_Adj[i, u, u] = 1.0  # Self loops

class DenseGCN(nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats):
        super().__init__()
        self.conv1 = nn.Linear(in_feats, hidden_feats)
        self.conv2 = nn.Linear(hidden_feats, hidden_feats)
        self.fc = nn.Linear(hidden_feats, out_feats)
        
    def forward(self, feats, adj):
        x = torch.bmm(adj, feats)
        # Use inplace=True to prevent duplicate allocation
        x = F.relu(self.conv1(x), inplace=True)
        x = torch.bmm(adj, x)
        # Use inplace=True to prevent duplicate allocation
        x = F.relu(self.conv2(x), inplace=True)
        x = x.sum(dim=1)
        return self.fc(x)

# 1. Benchmark CPU
print("\n--- Benchmarking CPU (Full-Batch, Hidden Feats = 400, Memory = 2.88 GB) ---")
device_cpu = torch.device('cpu')
model_cpu = DenseGCN(30, 400, 12).to(device_cpu)
optimizer_cpu = torch.optim.Adam(model_cpu.parameters(), lr=0.001)

# Warmup
_ = model_cpu(dense_X.to(device_cpu), dense_Adj.to(device_cpu))

start_cpu = time.time()
for epoch in range(1):
    optimizer_cpu.zero_grad()
    out = model_cpu(dense_X.to(device_cpu), dense_Adj.to(device_cpu))
    loss = F.binary_cross_entropy_with_logits(out, dense_y.to(device_cpu), weight=dense_w.to(device_cpu))
    loss.backward()
    optimizer_cpu.step()
end_cpu = time.time()
duration_cpu = end_cpu - start_cpu
print(f"Dense CPU GCN (1 epoch): {duration_cpu:.2f}s | Throughput: {batch_size / duration_cpu:.2f} mol/s")

# 2. Benchmark MPS GPU
print("\n--- Benchmarking MPS GPU (Full-Batch, Hidden Feats = 400, Memory = 2.88 GB) ---")
device_mps = torch.device('mps')
model_mps = DenseGCN(30, 400, 12).to(device_mps)
optimizer_mps = torch.optim.Adam(model_mps.parameters(), lr=0.001)

# Warmup
_ = model_mps(dense_X.to(device_mps), dense_Adj.to(device_mps))

start_mps = time.time()
for epoch in range(1):
    optimizer_mps.zero_grad()
    out = model_mps(dense_X.to(device_mps), dense_Adj.to(device_mps))
    loss = F.binary_cross_entropy_with_logits(out, dense_y.to(device_mps), weight=dense_w.to(device_mps))
    loss.backward()
    optimizer_mps.step()
end_mps = time.time()
duration_mps = end_mps - start_mps
print(f"Dense MPS GPU GCN (1 epoch): {duration_mps:.2f}s | Throughput: {batch_size / duration_mps:.2f} mol/s")

print(f"\n==========================================")
print(f"Dense GCN GPU Speedup over CPU: {duration_cpu / duration_mps:.2f}x")
print(f"==========================================")


Preparing dataset for full-batch Dense GCN...



--- Benchmarking CPU (Full-Batch, Hidden Feats = 400, Memory = 2.88 GB) ---


Dense CPU GCN (1 epoch): 2.90s | Throughput: 2149.92 mol/s

--- Benchmarking MPS GPU (Full-Batch, Hidden Feats = 400, Memory = 2.88 GB) ---


Dense MPS GPU GCN (1 epoch): 1.28s | Throughput: 4866.79 mol/s

Dense GCN GPU Speedup over CPU: 2.26x
